# ⚡ 05. Thuật toán FP-Growth & Khai phá Contrast Set (So sánh với Apriori)

**Thành viên thực hiện:** Thành viên 1 (ARM Lead)

Mục tiêu:
- Áp dụng thuật toán FP-Growth trên cùng dữ liệu.
- So sánh hiệu năng đối đầu trực tiếp: Thời gian thực thi & Bộ nhớ (Apriori vs FP-Growth).
- Khai phá Contrast Set (Tập tương phản): Tìm luật chỉ có ở trận Thắng mà KHÔNG có ở trận Thua.
- Khám phá 'Hidden Meta' (Tần suất thấp nhưng Lift và Confidence cực cao).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
from src.arm_miner import AssociationRuleMiner
import warnings
warnings.filterwarnings('ignore')

## 1. Nạp dữ liệu

In [ ]:
pkl_path = '../data/processed/bin_win.pkl'
csv_path = '../data/processed/bin_win.csv'

try:
    bin_win = pd.read_pickle(pkl_path)
    bin_lose = pd.read_pickle('../data/processed/bin_lose.pkl')
except Exception as e:
    print(f'Nạp pickle gặp cảnh báo ({e}), chuyển sang đọc file CSV an toàn.')
    bin_win = pd.read_csv(csv_path)
    bin_lose = pd.read_csv('../data/processed/bin_lose.csv')

target_cols = [c for c in bin_win.columns if c.startswith('Champ_') or c.startswith('Item_')]
bin_win = bin_win[target_cols]
bin_lose = bin_lose[[c for c in bin_lose.columns if c in target_cols]]

miner = AssociationRuleMiner(min_support=0.03, min_confidence=0.5, min_lift=1.2, max_len=3)
print(f'Đã nạp ma trận nhị phân: Win {bin_win.shape}, Lose {bin_lose.shape}')

## 2. Chạy FP-Growth

In [ ]:
t0 = time.time()
frequent_itemsets_fp, rules_fp, timing_fp = miner.run_fpgrowth(bin_win, win_label=True)
fp_time = time.time() - t0

print(f'⚡ Thời gian chạy FP-Growth: {fp_time:.2f} giây')
print(f'📦 Số tập mục phổ biến: {len(frequent_itemsets_fp)}')
print(f'📋 Số luật kết hợp: {len(rules_fp)}')

## 3. So sánh Hiệu năng Đối đầu (Apriori vs FP-Growth)

In [ ]:
# So sánh trực tiếp hiệu năng Apriori vs FP-Growth
sample_data = bin_win.sample(n=min(1500, len(bin_win)), random_state=42)
comp_df = miner.compare_algorithms(sample_data)
print(comp_df)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comp_df, x='Metric', y='Apriori', color='coral', label='Apriori', alpha=0.8)
sns.barplot(data=comp_df, x='Metric', y='FP-Growth', color='teal', label='FP-Growth', alpha=0.8)
ax.set_title('So sánh Apriori vs FP-Growth')
ax.legend()
plt.show()

## 4. Khai phá Tập Tương Phản (Contrast Set Mining: Win vs Lose)

In [ ]:
contrast_rules = miner.contrast_set_mining(bin_win, bin_lose)
print(f'Số luật đặc trưng thắng (Win-defining rules): {len(contrast_rules)}')
if not contrast_rules.empty:
    display(contrast_rules.head(10))

## 5. Khai phá Hidden Meta (Luật ít người chơi nhưng tỷ lệ thắng đột biến)

In [ ]:
hidden_meta = miner.find_hidden_meta(rules_fp, max_support=0.06, min_lift=1.6, min_confidence=0.65)
print(f'Số tổ hợp Hidden Meta phát hiện được: {len(hidden_meta)}')
miner.format_rules(hidden_meta, top_n=10)

## 6. Tổng kết phần ARM
- **FP-Growth** vượt trội hoàn toàn về tốc độ so với Apriori nhờ cấu trúc FP-Tree nén dữ liệu và không cần sinh ứng viên lặp đi lặp lại.
- **Contrast Set Mining** đã tách biệt thành công các tổ hợp tướng chỉ xuất hiện ở nhóm thắng.
- Đã phát hiện ra các cụm Hidden Meta tiềm năng.